### 导入包

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import plotly.express as px

### 读取数据

1.读取用户信息表、银行账单表以及信用卡账单表

In [ ]:
userinfo_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\userinfo.csv")
userinfo_df.head(5)

2.读取逾期表，并将第一列删掉

In [ ]:
overdue_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\overdue.csv")
overdue_df.drop(['Unnamed: 0'],axis=1,inplace=True)
overdue_df.columns=['new_user_id','违约','分数']
overdue_df

3.读取银行流水表

In [ ]:
bank_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bank.csv")
bank_df.head(5)

4.读取银行账单

In [ ]:
bill_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bill.csv")
bill_df

### 离散变量编码

3.统计用户信息表中“职业”变量有几个属性，户口类型有几个属性，并将这些属性用顺序编码的方式来进行编码，婚姻，学历等同理

In [ ]:
## 将离散变量转换为字典
def todict(col):
    res={}
    cnt=0
    for e in col.unique():
        res[e]=cnt
        cnt+=1
    return res

In [ ]:
userinfo_df=userinfo_df.replace({'性别':{'女':0,'男':1,'未知':2}})

In [ ]:
marriage_map = {'未婚': 0, '已婚': 1, '离异': 2, '丧偶': 3,'其他情况':4}
hukou_map = {'农业户口': 0, '非农业户口': 1, '集体户口': 2, '家庭户口': 3, '其他户口': 4}

userinfo_df = userinfo_df.replace({'婚姻状态': marriage_map, '户口类型': hukou_map})

In [ ]:
dict1=todict(userinfo_df['教育程度'])
userinfo_df=userinfo_df.replace(dict1)

In [ ]:
userinfo_df["职业"] = userinfo_df["职业"].map({'公职':0,'企业':1,'事业单位':2,'无业':3,'学生':4})

In [ ]:
userinfo_df

### 数据三值

统计用户信息表中的缺失值，并打印输出

In [ ]:
userinfo_df.isna().sum()

### 数据合并

In [ ]:
occupation = userinfo_df[["new_user_id","职业"]]
default = overdue_df[["new_user_id","分数"]]
default1 = overdue_df[["new_user_id","违约"]]
fenshu_data = pd.merge(occupation, default, how="outer", on="new_user_id")
data = pd.merge(fenshu_data, default1, how="outer", on="new_user_id")
data

### 相关性分析

In [ ]:
print(userinfo_df.isnull().sum())

def corr_analysis(data):
    corr = data.corr()
    
    fig = px.imshow(corr,
                    x=corr.columns,
                    y=corr.index,
                    color_continuous_scale='YlGnBu',
                    zmin=-1,
                    zmax=1,## 其实没啥影响，也能画，有了之后更标准化，默认0-1
                    title='相关性热力图')
    # 要有xy轴，z轴上下限，默认主题
    fig.update_layout(
        width=800,
        height=700 ## 控制绘图的大小
    )
    
    fig.show()

corr_analysis(userinfo_df.drop(['new_user_id'], axis=1))

### 计算相关系数

In [ ]:
bank_df

In [ ]:
salary_data = bank_df[bank_df['工资收入标记'] == 1].groupby('new_user_id')['交易金额'].mean().reset_index()
salary_data.columns = ['new_user_id', '工资总收入']
merged_data = pd.merge(data, salary_data, on='new_user_id', how='inner')


# 描述性统计
print(merged_data['工资总收入'].describe())
print(merged_data['分数'].describe())

In [ ]:
corr_np=merged_data['工资总收入'].corr(merged_data['分数'])
print(corr_np)

array格式下，乘法还是一个向量，点乘是内积，对应元素相乘再相加，语法是np.dot(a,b)


In [ ]:
x = merged_data['工资总收入']
y = merged_data['分数']

x_mean = x.mean()
y_mean = y.mean()
print(x_mean)
print(y_mean)


In [ ]:
numerator = np.sum((x - x_mean) * (y - y_mean))
x_std = np.sqrt(np.sum((x - x_mean) ** 2))
y_std = np.sqrt(np.sum((y - y_mean) ** 2))

correlation = numerator / (x_std * y_std)
print(correlation)



## 线性回归方法

### 线性回归方法一：公式法

In [ ]:
def theta_0(x,y):
    a1=np.sum(x**2)*np.sum(y)
    a2=np.sum(x)*np.sum(y*x)
    a3=len(x)*np.sum(x**2)-(np.sum(x))**2
    return float((a1-a2)/a3)


In [ ]:
def theta_1(x,y):
    a1=len(x)*np.sum(y*x)
    a2=np.sum(x)*np.sum(y)
    a3=len(x)*np.sum(x**2)-(np.sum(x))**2
    return float((a1-a2)/a3)

In [ ]:
theta_0=theta_0(x,y)
theta_1=theta_1(x,y)

print(theta_0)
print(theta_1)


### 线性回归方法二：格点法

<font color = red>4.使用网格搜索的方法编程计算𝜃_0和𝜃_1

我们进行如下设置：

斜率搜索范围都是  [−50,50]  ，步数  200  ，因此搜索步长自动计算为  (50−(−50))/200=0.5

截距搜索范围都是  [−50,50]  ，步数  200  ，因此搜索步长自动计算为  (50−(−50))/200=0.5

因此这里会计算  200×200=40000  次。不宜把步数设置太多，这将导致程序一直在执行。

 误差的方程是:$MSE=\sum_{i=1}^{n}(\theta_{0}+\theta_{1}x_{i}-y_{i})^{2}$

 误差的方程是:$MAE=\sum_{i=1}^{n}|\theta_{0}+\theta_{1}x_{i}-y_{i}|$


In [ ]:

x = merged_data_clean['工资总收入'].values
y = merged_data_clean['分数'].values

mse = float('inf')
best_theta0 = 0
best_theta1 = 0

theta0_range = np.arange(-50, 50, 0.5)
theta1_range = np.arange(-50, 50, 0.5)

for theta0 in theta0_range:
    y_pred = theta0 + theta1_range[:, np.newaxis] * x
    mse_values = np.mean((y_pred - y) ** 2, axis=1)
    min_index = np.argmin(mse_values)
    if mse_values[min_index] < mse:
        mse = mse_values[min_index]
        best_theta0 = theta0
        best_theta1 = theta1_range[min_index]

print(f"最佳 theta0: {best_theta0}")
print(f"最佳 theta1: {best_theta1}")
print(f"最小 MSE: {mse}")

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
# 将 x 转换为二维数组
x = merged_data_clean[['工资总收入']]  # 使用双中括号将 Series 转换为 DataFrame
y=  merged_data_clean[['分数']]

model.fit(x, y)

print(f"theta_0: {model.intercept_}")
print(f"theta_1: {model.coef_[0]}")



### 绘制曲线与散点图，计算对应误差值

梳理一下，顺序执行上面的代码，我们目前有：
1. 数据：x y
2. 用解方程得到的截距和斜率 theta_0 theta_1
3. 用搜索法得到的截距和斜率 best_theta_0 best_theta_1

接下来利用 theta_0 + theta_1 * x 以及 best_theta_0 + best_theta_1 * x 分别绘图，并且计算误差值。

误差的方程是:$MSE=\sum_{i=1}^{n}(\theta_{0}+\theta_{1}x_{i}-y_{i})^{2}$

<font color = red>5.绘制回归曲线与数据的散点图，并计算出这种情况下对应的误差值

In [ ]:
px.scatter(x,y)

In [ ]:
df = pd.DataFrame({
    '工资总收入': x,
    '分数': y,
    '解方程法预测': theta_0 + theta_1 * x,
    '搜索法预测': best_theta0 + best_theta1 * x
})
df

In [ ]:
fig = px.scatter(df, x='工资总收入', y='分数', title='工资总收入与分数的回归分析')
fig.add_scatter(x=df['工资总收入'], y=df['解方程法预测'], 
                mode='lines', name='解方程法回归线')
fig.add_scatter(x=df['工资总收入'], y=df['搜索法预测'], 
                mode='lines', name='搜索法回归线')
fig.update_layout(width=800, height=600)
fig.show()
mse_eq = ((df['分数'] - df['解方程法预测'])**2).mean()
mse_search = ((df['分数'] - df['搜索法预测'])**2).mean()

print(f"解方程法的MSE: {mse_eq}")
print(f"搜索法的MSE: {mse_search}")

### 敏感性分析：扰动参数观察误差值

所谓敏感性分析，就是将某个值固定，调整其他值，看目标函数的变化。做一个后验分析

1. 实验1：固定$\theta_{1}=0.23$，将$\theta_{0}$在[60,80]中变化，看MSE变化
2. 实验2：固定$\theta_{0}=71.38$，将$\theta_{1}$在[-1,1]中变化，看MSE变化

<font color = red>6.做一个敏感性分析，即对𝜃_0和𝜃_1进行扰动，注意观察扰动后的误差值是否是增大的

In [ ]:
theta_0_original = 71.38
theta_1_original = 0.23

In [ ]:

theta_0_original = 71.38
theta_1_original = 0.23

# 实验1: 固定theta_1,变化theta_0
theta_0_range = np.linspace(60, 80, 100)
mse_values_1 = []

for theta_0 in theta_0_range:
    y_pred = theta_0 + theta_1_original * x
    mse = np.mean((y - y_pred)**2)
    mse_values_1.append(mse)

# 实验2: 固定theta_0,变化theta_1
theta_1_range = np.linspace(-1, 1, 100)
mse_values_2 = []

for theta_1 in theta_1_range:
    y_pred = theta_0_original + theta_1 * x
    mse = np.mean((y - y_pred)**2)
    mse_values_2.append(mse)

# 绘制结果
fig1 = px.line(x=theta_0_range, y=mse_values_1, 
               title='固定θ1=0.23,θ0变化时的MSE',
               labels={'x':'θ0', 'y':'MSE'})
fig1.show()

fig2 = px.line(x=theta_1_range, y=mse_values_2, 
               title='固定θ0=71.38,θ1变化时的MSE',
               labels={'x':'θ1', 'y':'MSE'})
fig2.show()

# 找出最小MSE对应的theta值
min_mse_1 = min(mse_values_1)
min_theta_0 = theta_0_range[mse_values_1.index(min_mse_1)]

min_mse_2 = min(mse_values_2)
min_theta_1 = theta_1_range[mse_values_2.index(min_mse_2)]

print(f"实验1最小MSE: {min_mse_1:.4f}, 对应的θ0: {min_theta_0:.4f}")
print(f"实验2最小MSE: {min_mse_2:.4f}, 对应的θ1: {min_theta_1:.4f}")